In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

drive_path = "/content/drive/MyDrive/PROJECT_FSD50K"

local_path = "/content/fsd50k_local"

!unzip -q {drive_path}/FSD50K.metadata.zip -d {local_path}
!unzip -q {drive_path}/FSD50K.ground_truth.zip -d {local_path}
!7z x {drive_path}/FSD50K.eval_audio.zip -o{local_path} -y > /dev/null
!7z x {drive_path}/FSD50K.dev_audio.zip -o{local_path} -y > /dev/null

Mounted at /content/drive


In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm
from collections import Counter

from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, average_precision_score, classification_report



## **Load the FSD50K Development Data**

In this step, we load the main files needed for the baseline experiments.  
First, we define the paths to the development split, the vocabulary file, and the audio folder. Then, we read the metadata into pandas DataFrames.

After that, we create the full file path for each audio clip and check whether the corresponding `.wav` file exists in the audio directory. We keep only the rows with valid audio files to avoid errors later in the pipeline.

Finally, we transform the `mids` and `labels` columns into lists. This is necessary because FSD50K is a multi-label dataset, which means that one audio clip can contain more than one sound class.

In [17]:
# Define the base directory of the local FSD50K dataset
BASE_DIR = "/content/fsd50k_local"
# Define the base directory of the local FSD50K dataset
DEV_CSV = os.path.join(BASE_DIR, "FSD50K.ground_truth", "dev.csv")
# Define the path to the development ground truth CSV file
VOCAB_CSV = os.path.join(BASE_DIR, "FSD50K.ground_truth", "vocabulary.csv")
# Define the directory that contains the development audio files
AUDIO_DIR = os.path.join(BASE_DIR, "FSD50K.dev_audio")

# Load the development metadata
dev_df = pd.read_csv(DEV_CSV)
# Load the vocabulary file; the file has no header, so header=None is needed
vocab_df = pd.read_csv(VOCAB_CSV, header=None)
# Assign clear column names to the vocabulary DataFrame
vocab_df.columns = ["idx", "label_name", "mid"]

# Create the full path for each audio file using its filename
dev_df["path"] = dev_df["fname"].apply(lambda x: os.path.join(AUDIO_DIR, f"{x}.wav"))
# Check whether each audio file exists in the directory
dev_df["exists"] = dev_df["path"].apply(os.path.exists)
# Keep only the rows for which the audio file is available
dev_df = dev_df[dev_df["exists"]].copy()

# Convert the 'mids' column from a comma-separated string into a list
# This is useful because one audio clip may have several labels
dev_df["mid_list"] = dev_df["mids"].apply(lambda x: str(x).split(","))
# Convert the human-readable labels into a list as well
dev_df["label_list"] = dev_df["labels"].apply(lambda x: str(x).split(","))

print(dev_df.shape)
display(dev_df.head())

(40966, 8)


,fname,labels,mids,split,path,exists,mid_list,label_list
0,64760,"Electric_guitar,Guitar,Plucked_string_instrume...","/m/02sgy,/m/0342h,/m/0fx80y,/m/04szw,/m/04rlf",train,/content/fsd50k_local/FSD50K.dev_audio/64760.wav,True,"[/m/02sgy, /m/0342h, /m/0fx80y, /m/04szw, /m/0...","[Electric_guitar, Guitar, Plucked_string_instr..."
1,16399,"Electric_guitar,Guitar,Plucked_string_instrume...","/m/02sgy,/m/0342h,/m/0fx80y,/m/04szw,/m/04rlf",train,/content/fsd50k_local/FSD50K.dev_audio/16399.wav,True,"[/m/02sgy, /m/0342h, /m/0fx80y, /m/04szw, /m/0...","[Electric_guitar, Guitar, Plucked_string_instr..."
2,16401,"Electric_guitar,Guitar,Plucked_string_instrume...","/m/02sgy,/m/0342h,/m/0fx80y,/m/04szw,/m/04rlf",train,/content/fsd50k_local/FSD50K.dev_audio/16401.wav,True,"[/m/02sgy, /m/0342h, /m/0fx80y, /m/04szw, /m/0...","[Electric_guitar, Guitar, Plucked_string_instr..."
3,16402,"Electric_guitar,Guitar,Plucked_string_instrume...","/m/02sgy,/m/0342h,/m/0fx80y,/m/04szw,/m/04rlf",train,/content/fsd50k_local/FSD50K.dev_audio/16402.wav,True,"[/m/02sgy, /m/0342h, /m/0fx80y, /m/04szw, /m/0...","[Electric_guitar, Guitar, Plucked_string_instr..."
4,16404,"Electric_guitar,Guitar,Plucked_string_instrume...","/m/02sgy,/m/0342h,/m/0fx80y,/m/04szw,/m/04rlf",train,/content/fsd50k_local/FSD50K.dev_audio/16404.wav,True,"[/m/02sgy, /m/0342h, /m/0fx80y, /m/04szw, /m/0...","[Electric_guitar, Guitar, Plucked_string_instr..."


## **Select the Most Frequent Classes and Build the Baseline Split**

In this step, we identify the most frequent sound classes in the development set.  
We first count how often each class appears in the dataset and then keep only the 20 most common classes to build our baseline. This reduces the complexity of the problem and makes the first baseline model easier to train and evaluate.

Next, we filter the labels of each audio clip so that only the selected classes remain. We also remove the clips that no longer contain any of these classes after filtering.

Finally, we separate the filtered data into training and validation subsets using the official `split` column provided in the dataset.

In [18]:
# Create a flat list with all class IDs (mids) from the dataset
all_mids = [mid for mids in dev_df["mid_list"] for mid in mids]
# Count how many times each class appears
mid_counts = Counter(all_mids)
# Select the 20 most frequent classes
top_mids = [mid for mid, _ in mid_counts.most_common(20)]
# Print the selected top classes
print(top_mids)

# For each audio clip, keep only the labels that belong to the top 20 classes
dev_df["mid_list_filtered"] = dev_df["mid_list"].apply(
    lambda mids: [m for m in mids if m in top_mids]
)

# Remove clips that do not contain any of the selected classes
baseline_df = dev_df[dev_df["mid_list_filtered"].apply(len) > 0].copy()
# Split the filtered dataset into training and validation sets
# using the official split provided by FSD50K
train_df = baseline_df[baseline_df["split"] == "train"].copy()
val_df = baseline_df[baseline_df["split"] == "val"].copy()

# Print the size of the training and validation sets
print(train_df.shape)
print(val_df.shape)

['/m/04rlf', '/m/04szw', '/t/dd00071', '/m/09l8g', '/m/0jbk', '/m/0l14md', '/m/07yv9', '/m/085jw', '/m/0l14_3', '/m/0fx80y', '/m/0342h', '/m/01280g', '/m/09x0r', '/m/05148p4', '/m/012f08', '/m/0838f', '/m/07pp_mv', '/m/026t6', '/m/015p6', '/m/04k94']
(26894, 9)
(2933, 9)


## **Create Smaller Training and Validation Subsets**

In this phase, we reduce the size of the training and validation sets by selecting subsets from each division that maintain roughly the same ratio between the training and validation sets as the original set.  
This is useful for the baseline experiments because it decreases the computational cost and allows faster feature extraction and model training.
  
Finally, the indices are reset to keep both DataFrames clean and consistent.

In [19]:
# 5000 samples from the training set
train_df = train_df.sample(5000, random_state=42)
# 500 samples from the validation set
val_df = val_df.sample(500, random_state=42)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

## **Load Audio Files with a Fixed Length**

This function loads an audio file and applies a simple preprocessing step to standardize its format.  
All audio clips are converted to mono. Since this project focuses on sound content classification rather than spatial localization, the information contained in the left and right channels is not essential for the baseline models.
All audio clips are and resampled to 16 kHz, which reduces computational cost and ensures that all inputs have the same sampling rate.

The function also forces every clip to have the same duration of 5 seconds.  
If an audio file is longer than 5 seconds, it is truncated. If it is shorter, zero-padding is added at the end. This is important because machine learning models require inputs with a consistent size.

In [20]:
def load_audio_fixed(filepath, sr=16000, duration=10):
    # Load the audio file
    # sr=16000 resamples the signal to 16 kHz
    # mono=True converts the audio to mono
    y, sr = librosa.load(filepath, sr=sr, mono=True)

    # Compute the target number of samples
    target_len = sr * duration

    # If the audio is longer than the target length, cut it
    if len(y) > target_len:
        y = y[:target_len]
    else:
        # If the audio is shorter, add zeros at the end
        y = np.pad(y, (0, target_len - len(y)))

    # Return the processed waveform and the sampling rate
    return y, sr

## **Extract Basic Handcrafted Audio Features**

This function extracts a set of basic handcrafted features from each audio clip.  
The objective is to transform the raw waveform into a fixed numerical representation that can be used as input for the first baseline model.

Since machine learning models such as MLPs cannot work directly with raw audio in this baseline setting, we summarize each clip through a set of descriptors that capture different acoustic properties of the signal. In this case, we use **MFCCs, Zero Crossing Rate, Spectral Centroid, and Chroma**.

### **Why do we use these features?**

These features were selected because they provide complementary information about the audio signal:

- **MFCCs (Mel-Frequency Cepstral Coefficients)** describe the general spectral shape of the sound in a compact way. They are widely used in audio classification because they capture timbral information and are especially useful for distinguishing different types of sound events.
- **Zero Crossing Rate (ZCR)** measures how often the waveform changes sign. This can help differentiate signals with noisier or more percussive behavior from smoother and more tonal sounds.
- **Spectral Centroid** represents the “center of mass” of the spectrum. It gives an indication of how bright or sharp a sound is, which can be useful to separate low-frequency sounds from high-frequency ones.
- **Chroma** groups spectral energy into 12 pitch-related bins. Although it is often used in music analysis, it can still provide useful harmonic information for certain sound classes, especially those containing pitched or tonal components.

### **Why do we compute the mean and standard deviation?**

Each feature is calculated frame by frame across time, so the output is not a single value but a sequence of values.  
To obtain a fixed-length representation for every audio clip, we summarize each feature by computing:

- the **mean**, which describes its average behavior over time
- the **standard deviation**, which measures how much it changes during the clip

This gives a compact representation of the audio while preserving both central tendency and variability.

### **Why do we use a `try-except` block?**

The `try-except` structure is included to prevent the pipeline from stopping if a specific audio file cannot be processed.  
If an error occurs, the function returns `None`, and that sample can be skipped later.

In [21]:
def extract_handcrafted_features_basic(filepath, sr=16000, n_mfcc=13, duration=10):
    try:
        y, sr = load_audio_fixed(filepath, sr=sr, duration=duration)

        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
        zcr = librosa.feature.zero_crossing_rate(y)
        spec_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)

        features = np.concatenate([
            mfcc.mean(axis=1), mfcc.std(axis=1),
            zcr.mean(axis=1), zcr.std(axis=1),
            spec_centroid.mean(axis=1), spec_centroid.std(axis=1),
            chroma.mean(axis=1), chroma.std(axis=1)
        ])

        return features

    except Exception:
        return None

## **Extract Improved Handcrafted Audio Features**

This function extends the previous feature extraction approach by adding more descriptive spectral and temporal information.  
The goal is to build a richer representation of each audio clip and evaluate whether a more informative input space improves the performance of the MLP baseline.

In addition to the features used in the basic version, this improved version includes:

- **Delta MFCCs**, which capture how the MFCC values change over time
- **Delta-Delta MFCCs**, which capture the acceleration of those changes
- **Spectral Bandwidth**, which measures the spread of the spectrum around its centroid
- **Spectral Rolloff**, which indicates the frequency below which most of the spectral energy is concentrated
- **RMS Energy**, which represents the average energy or intensity of the signal

These additional features are useful because they provide information that is not fully captured by the basic set alone.  
In particular, they help describe **temporal dynamics**, **spectral spread**, and **signal energy**, which may improve the discrimination between different environmental sounds.

In [22]:
def extract_handcrafted_features_improved(filepath, sr=16000, n_mfcc=13, duration=10):
    try:
        y, sr = load_audio_fixed(filepath, sr=sr, duration=duration)

        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
        mfcc_delta = librosa.feature.delta(mfcc)
        mfcc_delta2 = librosa.feature.delta(mfcc, order=2)

        zcr = librosa.feature.zero_crossing_rate(y)
        spec_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
        spec_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
        spec_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        rms = librosa.feature.rms(y=y)

        features = np.concatenate([
            mfcc.mean(axis=1), mfcc.std(axis=1),
            mfcc_delta.mean(axis=1), mfcc_delta.std(axis=1),
            mfcc_delta2.mean(axis=1), mfcc_delta2.std(axis=1),
            zcr.mean(axis=1), zcr.std(axis=1),
            spec_centroid.mean(axis=1), spec_centroid.std(axis=1),
            spec_bandwidth.mean(axis=1), spec_bandwidth.std(axis=1),
            spec_rolloff.mean(axis=1), spec_rolloff.std(axis=1),
            chroma.mean(axis=1), chroma.std(axis=1),
            rms.mean(axis=1), rms.std(axis=1)
        ])

        return features

    except Exception:
        return None

## **Build the Feature Matrix**

This function applies the selected feature extraction method to all audio files in a DataFrame.  
Its purpose is to transform the dataset from a collection of audio paths into a numerical feature matrix that can be used as input for the baseline model.

The function processes each audio file one by one, extracts its feature vector, and stores the valid results in a list.  
At the same time, it keeps track of the rows that were successfully processed. This is important because some files may fail during feature extraction, and these samples must be removed to keep the features and labels aligned.

At the end, the extracted features are converted into a NumPy array `X`, and a cleaned version of the original DataFrame is returned with only the valid rows.

In [23]:
def build_feature_matrix(df, feature_fn):
    # List to store the extracted feature vectors
    features = []
    # List to store the indices of the rows that were processed correctly
    valid_indices = []
    # Iterate over all rows in the DataFrame
    # tqdm is used to display a progress bar
    for i, row in tqdm(df.iterrows(), total=len(df)):
        # Extract features from the audio file path
        feat = feature_fn(row["path"])
        # Keep only valid feature vectors
        if feat is not None:
            features.append(feat)
            valid_indices.append(i)

    # Convert the list of features into a NumPy array
    X = np.array(features)
    # Keep only the rows that were successfully processed
    df_valid = df.loc[valid_indices].reset_index(drop=True)
    # Return the feature matrix and the filtered DataFrame
    return X, df_valid

## **Encode the Multi-Label Targets**

Since FSD50K is a **multi-label dataset**, each audio clip can belong to more than one class at the same time.  
For this reason, the target labels cannot be represented with a single category per sample. Instead, they must be converted into a binary multi-label format.

The `MultiLabelBinarizer` is used to transform the list of class identifiers into a matrix where:

- each **row** represents one audio clip
- each **column** represents one class
- a value of **1** means that the class is present in the clip
- a value of **0** means that the class is absent

Here, the binarizer is initialized with `top_mids`, which ensures that the target matrix follows the same class order for all experiments.

In [24]:
mlb = MultiLabelBinarizer(classes=top_mids)

## **Evaluate the Multi-Label Predictions**

This function evaluates the performance of the model in the multi-label classification setting.  
The model first outputs probabilities for each class, and these probabilities are then converted into binary predictions using a decision threshold.

Three evaluation metrics are computed:

- **Micro-F1**, which aggregates all class decisions together and is more influenced by the most frequent classes
- **Macro-F1**, which computes the F1-score independently for each class and then averages the results, giving equal importance to all classes
- **mAP (mean Average Precision)**, which evaluates the ranking quality of the predicted probabilities and is widely used in multi-label problems

These three metrics provide a more complete view of performance, since they capture different aspects of the model’s behavior.

In [25]:
def evaluate_multilabel(y_true, y_prob, threshold=0.2):
    y_pred = (y_prob >= threshold).astype(int)

    micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    map_score = average_precision_score(y_true, y_prob, average="macro")

    return micro_f1, macro_f1, map_score

## **Train the Basic MLP Baseline**

In this step, we prepare the input data for the first baseline model and train the classifier.  
First, we extract the handcrafted features from the training and validation sets using the basic feature extraction function.

Next, we convert the multi-label targets into binary format with `MultiLabelBinarizer`. This produces a target matrix in which each column corresponds to one class and each row corresponds to one audio clip.

After that, we apply `StandardScaler` to normalize the feature values. This is important because the extracted features are not on the same numerical scale.

The classifier used here is a **One-vs-Rest MLP**, which trains one binary classifier for each class. The neural network contains two hidden layers with 128 and 64 neurons. This architecture was chosen as a simple but sufficiently flexible structure to model non-linear relationships between handcrafted audio features, while keeping the baseline computationally manageable. Early stopping is activated to avoid unnecessary training and reduce the risk of overfitting.

Finally, the trained model is used to generate class probabilities for the validation set, which will later be evaluated with the multi-label metrics.

In [26]:
# Extract the basic handcrafted features from the training and validation sets
X_train_basic, train_basic_df = build_feature_matrix(train_df, extract_handcrafted_features_basic)
X_val_basic, val_basic_df = build_feature_matrix(val_df, extract_handcrafted_features_basic)
# Convert the filtered class lists into binary multi-label format
y_train_basic = mlb.fit_transform(train_basic_df["mid_list_filtered"])
# transform is used on the validation set to keep the same class order
y_val_basic = mlb.transform(val_basic_df["mid_list_filtered"])
# This helps the MLP train more effectively because all features will have comparable scales
scaler_basic = StandardScaler()
X_train_basic = scaler_basic.fit_transform(X_train_basic)
X_val_basic = scaler_basic.transform(X_val_basic)

# OneVsRestClassifier trains one binary classifier per class
mlp_basic = OneVsRestClassifier(
    MLPClassifier(
        hidden_layer_sizes=(128, 64),   # two hidden layers
        max_iter=300,                   # maximum number of training iterations
        early_stopping=True,            # stop training early if validation score does not improve
        n_iter_no_change=10,            # for early stopping
        random_state=42
    )
)

# Train the baseline MLP on the standardized training features
mlp_basic.fit(X_train_basic, y_train_basic)

# Predict class probabilities for the validation set
# These probabilities will later be converted into binary predictions
y_prob_basic = mlp_basic.predict_proba(X_val_basic)

  0%|          | 19/5000 [00:01<06:11, 13.42it/s]/usr/local/lib/python3.12/dist-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
  7%|▋         | 36/500 [00:03<00:32, 14.35it/s]/usr/local/lib/python3.12/dist-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
100%|██████████| 500/500 [00:41<00:00, 11.90it/s]


## **Threshold Selection for the Basic MLP**

In multi-label classification, the model outputs a probability for each class rather than a final binary decision.  
To convert these probabilities into predicted labels, it is necessary to define a decision threshold.

In this step, we test several threshold values between 0.1 and 0.5. This allows us to observe how the final predictions change depending on how strict the decision rule is. Lower thresholds usually produce more positive predictions, while higher thresholds are more conservative.

The purpose of this comparison is to identify the threshold that gives the most balanced performance for the basic MLP baseline.  
The selected threshold will then be used in the final comparison of results, so that the evaluation is based on the most appropriate decision setting for this model.

In [27]:
for th in [0.1, 0.2, 0.3, 0.4, 0.5]:
    micro, macro, map_score = evaluate_multilabel(y_val_basic, y_prob_basic, threshold=th)
    print(f"Basic | th={th:.1f} | micro={micro:.4f} | macro={macro:.4f} | mAP={map_score:.4f}")

Basic | th=0.1 | micro=0.4865 | macro=0.3630 | mAP=0.3982
Basic | th=0.2 | micro=0.5400 | macro=0.3871 | mAP=0.3982
Basic | th=0.3 | micro=0.5416 | macro=0.3858 | mAP=0.3982
Basic | th=0.4 | micro=0.5274 | macro=0.3427 | mAP=0.3982
Basic | th=0.5 | micro=0.4937 | macro=0.2926 | mAP=0.3982


## **Train the Improved MLP Baseline**

In this step, we repeat the same training pipeline used for the basic MLP, but now with the improved handcrafted features.  
The purpose is to evaluate whether the richer audio representation leads to better multi-label classification performance.

As before, we first extract the features from the training and validation sets, then convert the target labels into binary multi-label format. After that, we standardize the input variables using `StandardScaler`, since the extracted features have different numerical ranges.

The classifier keeps the same architecture as the previous model, with two hidden layers of 128 and 64 neurons. This makes the comparison between the two baselines fairer, because the main difference comes from the feature set, not from changes in the model itself.

Finally, the trained model is used to predict class probabilities on the validation set. These probabilities will later be evaluated using different thresholds.

In [28]:
X_train_imp, train_imp_df = build_feature_matrix(train_df, extract_handcrafted_features_improved)
X_val_imp, val_imp_df = build_feature_matrix(val_df, extract_handcrafted_features_improved)

y_train_imp = mlb.fit_transform(train_imp_df["mid_list_filtered"])
y_val_imp = mlb.transform(val_imp_df["mid_list_filtered"])

scaler_imp = StandardScaler()
X_train_imp = scaler_imp.fit_transform(X_train_imp)
X_val_imp = scaler_imp.transform(X_val_imp)

mlp_improved = OneVsRestClassifier(
    MLPClassifier(
        hidden_layer_sizes=(128, 64),
        max_iter=300,
        early_stopping=True,
        n_iter_no_change=10,
        random_state=42
    )
)

mlp_improved.fit(X_train_imp, y_train_imp)
y_prob_imp = mlp_improved.predict_proba(X_val_imp)

  0%|          | 20/5000 [00:02<08:12, 10.10it/s]/usr/local/lib/python3.12/dist-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
  7%|▋         | 37/500 [00:03<00:47,  9.71it/s]/usr/local/lib/python3.12/dist-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
100%|██████████| 500/500 [01:00<00:00,  8.29it/s]


## **Threshold Selection for the Improved MLP**
In multi-label classification, the model outputs a probability for each class rather than a final binary decision.
To convert these probabilities into predicted labels, it is necessary to define a decision threshold.

In this step, we test several threshold values between 0.1 and 0.5. This allows us to observe how the final predictions change depending on how strict the decision rule is. Lower thresholds usually produce more positive predictions, while higher thresholds are more conservative.

The purpose of this comparison is to identify the threshold that gives the most balanced performance for the basic MLP baseline.
The selected threshold will then be used in the final comparison of results, so that the evaluation is based on the most appropriate decision setting for this model.

In [29]:
for th in [0.1, 0.2, 0.3, 0.4, 0.5]:
    micro, macro, map_score = evaluate_multilabel(y_val_imp, y_prob_imp, threshold=th)
    print(f"Improved | th={th:.1f} | micro={micro:.4f} | macro={macro:.4f} | mAP={map_score:.4f}")

Improved | th=0.1 | micro=0.5483 | macro=0.4342 | mAP=0.4435
Improved | th=0.2 | micro=0.6017 | macro=0.4611 | mAP=0.4435
Improved | th=0.3 | micro=0.5977 | macro=0.4344 | mAP=0.4435
Improved | th=0.4 | micro=0.5959 | macro=0.4115 | mAP=0.4435
Improved | th=0.5 | micro=0.5747 | macro=0.3682 | mAP=0.4435


## **Final Comparison of the Baseline Models**

In this final step, we compare the two MLP baselines using the threshold selected for each model after the validation analysis.  
The purpose is to evaluate whether the improved handcrafted feature set provides a measurable advantage over the basic one.

The results show that the improved MLP outperforms the basic MLP in all three evaluation metrics:

- **Micro-F1** increases from **0.5416** to **0.6017**
- **Macro-F1** increases from **0.3858** to **0.4611**
- **mAP** increases from **0.3982** to **0.4435**

These results suggest that the additional features included in the improved version provide a richer representation of the audio signal.

It is also important to note that the two models use different decision thresholds in the final comparison.  
This choice is justified because each model produces a different probability distribution, so the best threshold must be selected independently on the validation set.

Overall, the improved MLP provides a stronger baseline for the next stages of the project, where it will later be compared with more advanced models such as CNNs and AST.

In [30]:
micro_basic, macro_basic, map_basic = evaluate_multilabel(y_val_basic, y_prob_basic, threshold=0.3)
micro_imp, macro_imp, map_imp = evaluate_multilabel(y_val_imp, y_prob_imp, threshold=0.2)

results_baseline = pd.DataFrame([
    {
        "Model": "MLP basic",
        "Threshold": 0.3,
        "Micro-F1": micro_basic,
        "Macro-F1": macro_basic,
        "mAP": map_basic
    },
    {
        "Model": "MLP improved",
        "Threshold": 0.2,
        "Micro-F1": micro_imp,
        "Macro-F1": macro_imp,
        "mAP": map_imp
    }
])

results_baseline

,Model,Threshold,Micro-F1,Macro-F1,mAP
0,MLP basic,0.1,0.541646,0.385768,0.398242
1,MLP improved,0.2,0.601669,0.461147,0.443474
